In [1]:
from agents import Agent, Runner
from dotenv import load_dotenv
from pydantic import BaseModel
import gradio as gr


c:\Users\dinga\Desktop\AI-Project N01\Src\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
from pydantic import BaseModel

class JobAnalysis(BaseModel):
    resume_skills: list[str]
    required_skills: list[str]
    matching_skills: list[str]
    missing_skills: list[str]

In [4]:
#Create first agent


agent = Agent(
    name ="Job-analysis agent",
    instructions = 
    """
    Analyzes the student’s resume and target job description,

    Extracts skills, education, projects, and work experience,

    Identifies job requirements and qualifications,

    Finds matches and potential skill gaps""",

    model ="gpt-4o-mini"
)

In [5]:

resume = """ 
I am a dot net developer with 5+ years expeirence,
I am profficient in C# and react for front end, also SQU and Mongo db database

"""
job_description = """
 we are looking for a dot net developer with 4+ years of expirience also with anglular expert
"""
input_string =  f""" resume:
{resume} 
job description:
 {job_description}.
 """

In [6]:
result1 = await Runner.run(agent, input_string)
print(result1.final_output)

### Resume Analysis

**Skills:**
- **Programming Languages:** C#
- **Frontend Technologies:** React
- **Databases:** SQL, MongoDB

**Experience:**
- 5+ years as a .NET developer

---

### Job Description Requirements

**Qualifications:**
- .NET development experience: 4+ years
- Expertise in Angular
- Additional skills in relevant technologies may be desired

---

### Match Analysis

**Matches:**
- The candidate meets the experience requirement (5+ years).
- Proficient in a relevant programming language (.NET with C#).

**Skill Gaps:**
- Lack of experience with Angular, which is specifically required by the job description.
- No mention of experience with additional technologies commonly used alongside .NET (e.g., ASP.NET, Entity Framework) or other relevant frontend technologies.

---

### Recommendations
- **Upgrading Skills:** 
  - Consider learning Angular to meet the job qualifications.
- **Highlighting Related Experience:**
  - If there are any projects or work experiences involv

In [7]:
agent =Agent[any](
    name="Interview-agent",
    instructions="""
    create interview questions,
    ask personalized questions to the job seaker
    
    """,
    model ="gpt-4o-mini"
)

In [8]:
result = await Runner.run(agent, "ASP.NET interview questions")
print(result.final_output)

Here are some ASP.NET interview questions that you can consider:

### Basic ASP.NET Questions
1. **What is ASP.NET?**
   - Can you explain the role of the .NET Framework in ASP.NET?

2. **What are the different types of ASP.NET applications?**
   - Can you differentiate between Web Forms and MVC?

3. **What is the Global.asax file?**
   - What are some common events you can handle in this file?

### ASP.NET MVC Questions
4. **What is MVC? Explain its components.**
   - Could you provide an example of how MVC works in an ASP.NET application?

5. **What are Action Filters in ASP.NET MVC?**
   - Can you explain the difference between Action filters, Result filters, and Exception filters?

6. **What are Routes in ASP.NET MVC?**
   - How can you customize routing?

### ASP.NET Web API Questions
7. **What is ASP.NET Web API?**
   - How does it differ from ASP.NET MVC?

8. **What are the advantages of using Web API?**
   - Can you explain REST and its principles?

### State Management
9. **Wh

In [ ]:
import os
import uuid
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# SETUP
# ============================================================

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file.")

client = OpenAI(api_key=api_key)


# ============================================================
# FILE READING HELPERS
# ============================================================

def read_uploaded_text(file_path):
    if not file_path:
        return ""

    file_path = str(file_path)
    ext = os.path.splitext(file_path)[1].lower()

    try:
        if ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()

        if ext == ".pdf":
            try:
                from pypdf import PdfReader
            except ImportError as exc:
                raise RuntimeError("Missing dependency: install with 'pip install pypdf'") from exc

            reader = PdfReader(file_path)
            pages = [page.extract_text() or "" for page in reader.pages]
            return "\n".join(pages)

        if ext == ".docx":
            try:
                from docx import Document
            except ImportError as exc:
                raise RuntimeError("Missing dependency: install with 'pip install python-docx'") from exc

            doc = Document(file_path)
            return "\n".join(paragraph.text for paragraph in doc.paragraphs)

        return ""

    except Exception as exc:
        raise ValueError(f"Could not read uploaded file: {os.path.basename(file_path)}. Please upload a .txt, .pdf, or .docx file.") from exc


# ============================================================
# TEXT TO SPEECH
# AI reads interview questions aloud
# ============================================================

def text_to_speech(text):
    try:
        audio_file = os.path.abspath(f"question_{uuid.uuid4().hex}.mp3")

        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="alloy",
            input=text
        ) as response:
            response.stream_to_file(audio_file)

        return audio_file

    except Exception as exc:
        print("TTS Error:", exc)
        return None


# ============================================================
# PERSONALIZED QUESTION GENERATION
# ============================================================

def build_personalized_question(resume_text, job_description):
    prompt = f"""
You are an interview coach.

Use the resume and job description below to generate ONE highly relevant interview question.
The question should help the candidate explain how their background matches the role.

Resume:
{resume_text[:4000]}

Job Description:
{job_description[:4000]}

Return only the interview question, no extra text.
"""

    try:
        response = client.responses.create(
            model="gpt-4o-mini",
            input=prompt
        )
        question = response.output_text.strip()
        if question:
            return question
    except Exception as exc:
        print("Question generation error:", exc)

    return (
        "Tell me about yourself and explain how your experience connects to this role and the requirements in the job description."
    )


# ============================================================
# START INTERVIEW
# ============================================================

def start_interview(resume_file, job_description_file):
    if resume_file is None:
        return (
            "Please upload your resume.",
            None,
            "❌ Resume missing"
        )

    if job_description_file is None:
        return (
            "Please upload the job description.",
            None,
            "❌ Job description missing"
        )

    try:
        resume_text = read_uploaded_text(resume_file)
        job_description = read_uploaded_text(job_description_file)
    except ValueError as exc:
        return (
            str(exc),
            None,
            "❌ File read error"
        )

    if not resume_text.strip() or not job_description.strip():
        return (
            "The uploaded files are empty or unreadable. Please upload valid text, PDF, or DOCX files.",
            None,
            "❌ Empty files"
        )

    question = build_personalized_question(resume_text, job_description)
    audio = text_to_speech(question)

    return (
        question,
        audio,
        "🟢 Interview started"
    )


# ============================================================
# TRANSCRIBE CANDIDATE AUDIO
# ============================================================

def transcribe_audio(audio_file):
    if audio_file is None:
        return None

    try:
        with open(audio_file, "rb") as audio:
            transcription = client.audio.transcriptions.create(
                model="gpt-4o-mini-transcribe",
                file=audio
            )
        return transcription.text

    except Exception as exc:
        print("Transcription Error:", exc)
        return None


# ============================================================
# SUBMIT ANSWER
# ============================================================

def submit_answer(audio_file):
    if audio_file is None:
        return (
            "",
            "❌ Record your answer first."
        )

    transcript = transcribe_audio(audio_file)

    if transcript is None:
        return (
            "",
            "❌ Could not transcribe recording."
        )

    return (
        transcript,
        "✅ Answer received"
    )


# ============================================================
# GET AI FEEDBACK
# ============================================================

def get_feedback(question, answer):
    if not answer or not answer.strip():
        return (
            "Please record and submit your answer first.",
            "❌ No answer available"
        )

    try:
        prompt = f"""
You are an AI Interview Coach.

Evaluate the candidate's interview answer.

Interview Question:
{question}

Candidate Answer:
{answer}

Give short and useful interview feedback.

Evaluate:
1. Relevance
2. Clarity
3. Technical/content quality
4. Communication
5. What was done well
6. What should be improved

Then give an improved example answer.

Keep the feedback concise and practical.
"""

        response = client.responses.create(
            model="gpt-4o-mini",
            input=prompt
        )

        feedback = response.output_text

        return (
            feedback,
            "⭐ Feedback generated"
        )

    except Exception as exc:
        print("Feedback Error:", exc)
        return (
            "Unable to generate feedback.",
            "❌ Feedback error"
        )


# ============================================================
# COMPACT CSS
# ============================================================

css = """

.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    padding: 6px !important;
}

h1 {
    font-size: 24px !important;
    margin: 2px !important;
}

h2 {
    font-size: 17px !important;
    margin: 2px !important;
}

footer {
    display: none !important;
}

"""


# ============================================================
# USER INTERFACE
# ============================================================

with gr.Blocks(
    title="AI Interview Coach"
) as demo:

    gr.Markdown(
        """
        # 🎤 AI Interview Coach
        **Upload → Start → Listen → Record → Submit → Feedback**
        """
    )

    with gr.Row():
        resume_file = gr.File(
            label="📄 Resume",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath",
            height=65
        )

        jd_file = gr.File(
            label="📋 Job Description",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath",
            height=65
        )

        start_button = gr.Button(
            "▶ Start Interview",
            variant="primary"
        )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 🤖 Interviewer")
            question_output = gr.Textbox(
                label="Question",
                lines=2,
                max_lines=3,
                interactive=False
            )
            question_voice = gr.Audio(
                label="🔊 Question Audio",
                autoplay=True,
                interactive=False
            )

        with gr.Column(scale=1):
            gr.Markdown("## 🎙️ Your Answer")
            voice_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Record Answer"
            )
            submit_button = gr.Button(
                "✅ Submit Answer",
                variant="primary"
            )

    with gr.Row():
        transcript_output = gr.Textbox(
            label="📝 Your Answer",
            lines=2,
            max_lines=3,
            interactive=False
        )

        status_output = gr.Textbox(
            label="Status",
            lines=1,
            interactive=False
        )

    feedback_button = gr.Button(
        "⭐ Get Interview Feedback",
        variant="secondary"
    )

    feedback_output = gr.Textbox(
        label="🧠 AI Coach Feedback",
        placeholder="Your interview feedback will appear here...",
        lines=5,
        max_lines=7,
        interactive=False
    )

    start_button.click(
        fn=start_interview,
        inputs=[resume_file, jd_file],
        outputs=[question_output, question_voice, status_output]
    )

    submit_button.click(
        fn=submit_answer,
        inputs=[voice_input],
        outputs=[transcript_output, status_output]
    )

    feedback_button.click(
        fn=get_feedback,
        inputs=[question_output, transcript_output],
        outputs=[feedback_output, status_output]
    )


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":
    demo.launch(inbrowser=True, css=css)


C:\Users\dinga\AppData\Local\Temp\ipykernel_20048\1154768386.py:231: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
